In [1]:
### Preamples ###
import numpy as np
import matplotlib.pyplot as plt
import random
from numba import njit

random.seed(13)

In [2]:
@njit
def heston_returns(V0, N, mu, theta, kappa, sigma_V, rho, h, n_sub):
    
    dt = h / n_sub
    sqrt_dt = dt ** 0.5
    c = (1 - rho ** 2) ** 0.5 
    
    returns = np.empty(N)
  
    Vt = V0
    log_ret = 0
    
    for n in range(N):
        log_ret = 0
        for _ in range(n_sub):
            z1 = np.random.normal(0, 1)
            z2 = np.random.normal(0, 1)
            
            dB1 = sqrt_dt * z1
            dB2 = sqrt_dt * (rho * z1 + c * z2) 
            
            Vt_pos = Vt if Vt > 0 else 0 # ensure variance is non-negative for the price update
            
            log_ret += (mu - 0.5 * Vt) * dt + Vt_pos ** 0.5 * dB1 # log return increment
            
            Vt = Vt_pos + kappa * (theta - Vt_pos)* dt + sigma_V * Vt_pos ** 0.5 * dB2  # variance process
            
        returns[n] = log_ret
        
    return returns 

In [3]:
def h_tilde(kappa, h):
    "h_tidle = (1 - exp(-kappa * h)) / kappa"
    return (1 - np.exp(-kappa * h)) / kappa

def lagm_cov_est(ret, m):
    
    N = len(ret)
    Ybar = np.mean(ret)
    
    cov = np.sum((ret[:N-m] - Ybar) * (ret[m:] - Ybar)) / (N - m)
    return cov

def estimate_kappa(ret, h , lag_M = 10):
    N = len(ret)
    
    lag1 = lagm_cov_est(ret, 1)
    
    kappa_sum = 0
    
    for m in range(2, lag_M + 1):
        lagm = lagm_cov_est(ret, m)
        
        if lagm == 0 or np.sign(lag1) != np.sign(lagm):
            continue
        
        ratio = lag1 / lagm
        if ratio <= 0 or not np.isfinite(np.log(ratio)):
            continue
        
        k_est = np.log(ratio) / ((m-1) * h)
        if not np.isfinite(k_est) or k_est <= 0:
            continue
        kappa_sum += k_est
        
    kappa = kappa_sum / (lag_M - 1)
    return kappa

def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])

def moments(mu, theta, kappa, sigma_V, rho, h):
    """Returns first five moments of the log returns of the Heston model."""
    ht = h_tilde(kappa, h)
    ek = np.exp(-kappa * h)
    
    # First moment (mean), follows from Theorem 2.1
    
    mean = h * (mu  - 0.5 * theta)
    
    # Second moment (variance), follows from Theorem 2.2
    
    var = theta * (h - ht) * ( (sigma_V**2)/ (4 * (kappa ** 2)) 
                              - (rho * sigma_V) / kappa) + theta * h
    
    # Third moment (lag-1 autocovariance), follows from Theorem 2.3
    
    lag1 = (((ht ** 2) * theta * (sigma_V)) / 2 ) * ( (sigma_V) / (4 * kappa) - rho)
    
    # Fourth moment (lag-2 autocovariance), follows from Theorem 2.4
    
    lag2 = ek * lag1
    
    
    # Fifth moment (Cross variance bewteen squared retursn and future returns),
    # follows from Theorem 2.5
    
    term1 = ((theta * (sigma_V ** 4) / ( 8 * kappa **3))) * ht * (h * ek - ht)
    
    term2 = ht**2 * ((theta * sigma_V ** 2) / (4 * kappa) * mu * h
                     - (theta ** 2 * sigma_V ** 2) / (8 * kappa) * h
                     - (theta * sigma_V ** 2 ) / (4 * kappa))
    
    term3  = (-rho * sigma_V / 2 * ht) * (
        (3 * sigma_V**2 / (2 * kappa**2) - 2 * rho * sigma_V / kappa)
        * theta * (h * ek - ht)
        + (2 * mu * theta - theta**2) * h * ht
)
    
    cross = term1 + term2 - term3
    
    return mean, var, lag1, lag2, cross

In [4]:
def estimate_heston_from_returns(ret, h, verbose= False):
    N = len(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = estimate_moments(ret)
    
    kappa = estimate_kappa(ret, h)
    
    ht = h_tilde(kappa, h)
    
    ek = np.exp(-kappa * h)
    
    dh = h * ek - ht
    
    theta = var_e / h  - (2* (h-ht) / (h * kappa * ht ** 2)) * lag1_e 
    
    mu = mean_e / h + theta / 2
    
    num   = (4*kappa*mean_e
             + (8*dh/(theta*ht**3))*lag1_e
             - 2*kappa*cross_e/lag1_e)
    den   = (theta*ht**2/(2*lag1_e) - dh/(kappa*ht))

    sigma_sq = num / den
    sigma_V = sigma_sq**0.5
    
    rho = np.clip(sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
                  -0.999, 0.999)
    
    if verbose:
        print( f"  κ̂    = {kappa:.4f}")
        print(f"  θ̂    = {theta:.4f}")
        print(f"  μ̂    = {mu:.4f}")
        print(f"  σ̂_V  = {sigma_V:.6f}")
        print(f"  ρ̂    = {rho:.6f}")
    

    return {'mu': mu, 'theta': theta, 'kappa': kappa,
            'sigma_V': sigma_V, 'rho': rho}
    

In [ ]:
def experiment_lag_M(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000)):
    """
    Tests sensitivity of kappa estimate to the lag limit M.
    """
    M_vals = [2, 3, 5, 7, 10, 15, 20, 30]

    print(f"{'M':>5s}  {'mean(κ)':>10s}  {'std(κ)':>10s}  "
          f"{'bias(κ)':>10s}  {'RMSE(κ)':>10s}  {'Failed':>8s}")
    print("-" * 60)

    for M in M_vals:
        kappa_ests = []
        failed     = 0

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            try:
                kappa = estimate_kappa(ret, h, lag_M=M)
                if np.isfinite(kappa) and kappa > 0:
                    kappa_ests.append(kappa)
                else:
                    failed += 1
            except Exception:
                failed += 1

        if len(kappa_ests) == 0:
            print(f"  {M:>3d}  {'all failed':>10s}  {'—':>10s}  "
                  f"{'—':>10s}  {'—':>10s}  {failed:>8d}")
            continue

        vals = np.array(kappa_ests)
        bias = vals.mean() - TRUE['kappa']
        rmse = np.sqrt(bias**2 + vals.var())

        print(f"  {M:>3d}  {vals.mean():>10.4f}  {vals.std():>10.4f}  "
              f"{bias:>10.4f}  {rmse:>10.4f}  {failed:>8d}")

In [8]:
TRUE = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0 = TRUE['theta']    # start in stationarity
h = 1.0              # annual observation interval
N = int(400_000)          # number of observations
n_sub = int(20)              # sub-steps per interval — critical for stability

experiment_lag_M(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000))

    M     mean(κ)      std(κ)     bias(κ)     RMSE(κ)    Failed
------------------------------------------------------------
    2      0.1007      0.0507      0.0007      0.0507         2
    3      0.1065      0.0350      0.0065      0.0356         0
    5      0.1006      0.0234      0.0006      0.0234         0
    7      0.1006      0.0171      0.0006      0.0171         0
   10      0.1007      0.0143      0.0007      0.0143         0
   15      0.1015      0.0105      0.0015      0.0106         0
   20      0.1003      0.0095      0.0003      0.0095         0
   30      0.1000      0.0071      0.0000      0.0071         0
